# ✅ Teste de Conexão — Proxy LLM
Execute cada célula em ordem para validar que tudo está funcionando.

In [1]:
# Célula 1 — Instalar dependências
!pip install -q langchain-anthropic langchain


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\mnsmferr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Célula 2 — Configuração
# Preencha com o seu token e o domínio do curso

PROXY_URL   = "https://interview-server-mocado.b60gda.easypanel.host/"   # ex: https://llm.seudominio.com
ALUNO_TOKEN = "xpto_aluno-01"                    # token que o professor forneceu

In [3]:
# Célula 3 — Health check (sem LLM, só testa se o servidor responde)
import requests

r = requests.get(f"{PROXY_URL}/health")
print(r.status_code)
print(r.json())

200
{'status': 'ok', 'anthropic_key_configured': True, 'alunos_registrados': 10}


In [4]:
# Célula 4 — Teste direto via requests (sem LangChain)
import requests, json

response = requests.post(
    f"{PROXY_URL}/v1/messages",
    headers={
        "Authorization": f"Bearer {ALUNO_TOKEN}",
        "Content-Type": "application/json",
        "anthropic-version": "2023-06-01",
    },
    json={
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 64,
        "messages": [
            {"role": "user", "content": "Responda apenas: conexão ok!"}
        ]
    }
)

data = response.json()
print(data["content"][0]["text"])

Conexão ok!


In [5]:
# Célula 5 — Teste via LangChain (forma que usaremos no curso)
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=128,
)

response = llm.invoke([HumanMessage(content="Olá! Confirme que a conexão via LangChain está funcionando.")])
print(response.content)

# Olá! 👋

Sim, a conexão está funcionando perfeitamente! 

Consegui receber sua mensagem e estou pronto para ajudá-lo. Sou o Claude, um assistente de IA criado pela Anthropic.

## Como posso ajudá-lo?

Estou disponível para:
- 💬 Responder perguntas
- 📝 Ajudar com escrita e edição
- 💻 Programação e desenvolvimento
- 🔍 


## ✅ Se as células 4 e 5 responderam sem erro — tudo pronto!

| Célula | O que testa |
|--------|-------------|
| 3 | Servidor no ar e API key configurada |
| 4 | Request HTTP direto ao proxy |
| 5 | Integração LangChain → Proxy → Claude |

---
## 📧 Servidor de E-mail — Testes de Conexão

Execute as células abaixo em ordem para validar o servidor de e-mail.

In [ ]:
# Célula 6 — Configuração do servidor de e-mail
import requests

EMAIL_URL = "https://interview-email-server.b60gda.easypanel.host/" # "http://localhost:8001"
EMAIL_TOKEN = "aluno-01"   # mesmo token usado no proxy LLM
EMAIL_PASSWORD = "1234"

In [18]:
# Célula 7 — Login e obtenção do bearer token
r = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_PASSWORD,
})
print(r.status_code, r.json())

# O servidor retorna o próprio token — usamos ele como bearer nas próximas chamadas
AUTH_HEADERS = {"Authorization": f"Bearer {EMAIL_TOKEN}"}

200 {'token': 'aluno-01', 'name': 'Aluno 1', 'email': 'aluno01@curso.ia'}


In [19]:
# Célula 8 — Meu perfil (/me)
r = requests.get(f"{EMAIL_URL}/me", headers=AUTH_HEADERS)
print(r.status_code)
print(r.json())

200
{'name': 'Aluno 1', 'email': 'aluno01@curso.ia', 'token': 'aluno-01'}


In [20]:
# Célula 9 — Caixa de entrada (/emails/inbox)
r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=AUTH_HEADERS)
inbox = r.json()
messages = inbox["messages"]
print(f"{inbox['count']} mensagem(ns) para {inbox['email']}\n")
for msg in messages:
    print(f"[{msg['id']}] {msg['date']}  De: {msg['from']}  |  {msg['subject']}")

0 mensagem(ns) para aluno01@curso.ia



In [21]:
# Célula 10 — Ler um e-mail específico (/emails/{id})
if not messages:
    print("Inbox vazia — envie um e-mail primeiro (célula 11).")
else:
    MESSAGE_ID = messages[0]["id"]
    r = requests.get(f"{EMAIL_URL}/emails/{MESSAGE_ID}", headers=AUTH_HEADERS)
    msg = r.json()
    print(f"De:      {msg['from']}")
    print(f"Assunto: {msg['subject']}")
    print(f"Data:    {msg['date']}")
    print(f"\n{msg['body']}")

Inbox vazia — envie um e-mail primeiro (célula 11).


In [22]:
# Célula 11 — Enviar um e-mail (/emails/send)
r = requests.post(f"{EMAIL_URL}/emails/send", headers=AUTH_HEADERS, json={
    "to": "xpto_aluno-02",
    "subject": "Teste de envio",
    "body": "Olá! Este é um e-mail de teste enviado pelo notebook.",
})
print(r.status_code)
print(r.json())

422
{'detail': 'Destinatário(s) não encontrado(s): xpto_aluno-02. O email não pôde ser enviado.'}


In [ ]:
# Célula 12 — E-mails enviados (/emails/sent)
r = requests.get(f"{EMAIL_URL}/emails/sent", headers=AUTH_HEADERS)
sent = r.json()
print(sent["messages"][0])  # inspeciona estrutura real

---
## 🤖 Agente de E-mail — LangChain

Um agente que entende linguagem natural e envia e-mails pelo servidor.
Execute as células abaixo em ordem.

In [ ]:
# Agente — Célula A: instalar langgraph (necessário para create_agent)
!pip install -q langgraph

In [52]:
# Agente — Célula B: login e definição da ferramenta send_email
import requests
from langchain_core.tools import tool
from typing import Optional

# Login no servidor de e-mail
_login = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_PASSWORD,
})
assert _login.status_code == 200, f"Login falhou: {_login.json()}"
_agent_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Login OK →", _login.json())


@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Use esta ferramenta quando o usuário pedir para enviar um e-mail.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: Um ou mais e-mails em cópia, separados por vírgula (ex: "aluno02@curso.ia, aluno03@curso.ia") — opcional
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc

    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_agent_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

Login OK → {'token': 'aluno-01', 'name': 'Aluno 1', 'email': 'aluno01@curso.ia'}


In [53]:
# Agente — Célula C: criar o agente
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

PROXY_URL   = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN = "xpto_aluno-01"
EMAIL_URL   = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN = "aluno-01"

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=512,
)

email_agent = create_agent(
    model=llm,
    tools=[send_email],
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Quando o usuário pedir para enviar um e-mail, use a ferramenta send_email. "
        "Os e-mails dos usuários seguem o padrão alunoXX@curso.ia (ex: aluno02@curso.ia). "
        "Se o usuário não informar algum campo obrigatório, pergunte antes de enviar."
    ),
)

print("Agente criado com sucesso.")

Agente criado com sucesso.


In [56]:
# Agente — Célula D: invocar com linguagem natural
# Altere o prompt abaixo como quiser
prompt = (
    "Manda um email para aluno10@curso.ia com cópia para aluno09@curso.ia, "
    "com título 'Reunião amanhã às 12h' e corpo: "
    "'Oi, passando para confirmar nossa reunião de amanhã às 10h. Abraços.'"
)

result = email_agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    config={"recursion_limit": 10},
)
print(result["messages"][-1].content)

Perfeito! E-mail enviado com sucesso para aluno10@curso.ia com cópia para aluno09@curso.ia, com o título 'Reunião amanhã às 12h' e o corpo da mensagem solicitado.
